In [20]:
mapping = {}

mapping["periodontal_abscess"] = {
    "false toothache",
    "gum pain",
    "pain when touched",
    "sensitivity when biting",
    "unusual taste",
    "salty-tasting fluid",
    "gum swelling with pus",
    "pain on palpation",
}

mapping["simple_cavities"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
    "pain to sour",
    "discomfort when brushing",
    "pain to stimuli",
}

mapping["acute_apical_periodontitis"] = {
    "pain when chewing",
    "pain when touched",
    "tooth feels higher",
    "mobile tooth",
    "discomfort on palpation",
    "pain on percussion",
}

mapping["chronic_apical_periodontitis"] = {
    "gum swelling",
    "salty-tasting fluid",
    "pressure when biting",
    "dull ache in the tooth",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

mapping["pericoronitis"] = {
    "continuous pain in the wisdom tooth area",
    "pain when chewing",
    "pain when swallowing",
    "cannot open mouth fully",
    "swelling over the wisdom tooth",
    "inflamed gum",
    "partially erupted wisdom tooth",
    "salty-tasting fluid",
}

mapping["reversible_pulpitis"] = {
    "short sharp pain",
    "pain to cold",
    "pain to sweet",
}

mapping["irreversible_pulpitis"] = {
    "spontaneous pain",
    "strong prolonged pain",
    "pain to cold",
    "pain to heat",
    "throbbing pain radiating to the ear",
    "slight pain on percussion",
    "tolerable sensitivity on palpation",
}

mapping["pulp_necrosis"] = {
    "spontaneous pain",
    "intense short pain",
    "pain to heat",
    "pressure when biting",
    "sensitivity to percussion",
    "sensitivity to palpation",
}

In [21]:
from dataclasses import dataclass, field
import random
from typing import List, Dict
from typing import Set

@dataclass
class Case:
    diagnosis_truth: str
    symptoms_truth: Set[str]
    revealed_symptoms: Set[str] = field(default_factory=set)


def init_case(mapping: Dict[str, Set[str]]) -> Case:
    """
    Select a random disease and return it with all associated symptoms.
    Compatible with the colleague's mapping: Dict[str, Set[str]].
    """
    disease_key = random.choice(list(mapping.keys()))
    symptoms = set(mapping[disease_key])
    return Case(
        diagnosis_truth=disease_key,
        symptoms_truth=symptoms,
    )


# Test
case = init_case(mapping)
print("Disease:", case.diagnosis_truth)
print("Symptoms:")
for s in case.symptoms_truth:
    print(f"  - {s}")



Disease: irreversible_pulpitis
Symptoms:
  - pain to cold
  - pain to heat
  - tolerable sensitivity on palpation
  - slight pain on percussion
  - throbbing pain radiating to the ear
  - spontaneous pain
  - strong prolonged pain


In [54]:
!pip -q install groq
from google.colab import userdata
import os

os.environ["GROQ_API_KEY2"] = userdata.get("GROQ_API_KEY2")

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY2"])

resp = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{"role": "user", "content": "Tell me a short joke about dentists."}],
)

print(resp.choices[0].message.content)


Here's one:

Why did the dentist become a baker?

Because he kneaded the dough! (get it?)


In [55]:
from dataclasses import dataclass, field
import json, re
import os
from groq import Groq
from typing import List, Dict, Set

client = Groq(api_key=os.environ["GROQ_API_KEY2"])


@dataclass
class State:
    summary: str = ""
    slots: dict = field(default_factory=dict)
    history: List[Dict[str, str]] = field(default_factory=list)


def build_patient_system_prompt(case: Case, max_new_clues: int = 1) -> str:
    """
    Build the system prompt for the virtual patient.
    Uses the ground-truth disease and its true symptoms (from case.symptoms_truth).
    """
    # acum symptoms_truth este un set de string-uri
    symptom_labels = sorted(case.symptoms_truth)
    symptoms_text = ", ".join(symptom_labels)

    return f"""
You are a VIRTUAL HUMAN PATIENT talking to a dental medicine student.
Your goal is to help the student practice identifying symptoms, NOT to tell them the diagnosis.

INTERNAL INFO (only for you, never reveal directly):
- True underlying disease (hidden diagnosis): {case.diagnosis_truth}
- Full list of your true symptoms (internal, do not list them all at once): {symptoms_text}

This internal symptom list is your ONLY source of truth.
You MUST NEVER claim (in your words) to have any pain, sensitivity,
discomfort, weird feeling or problem that is NOT in this list.

IMPORTANT:
The student will often use natural, informal language
(e.g. "does it hurt when you drink something cold?", "does it hurt when you chew on that side?",
"is your jaw clicking?", "do you feel pressure under your eyes?").
You should usually understand what symptom they refer to.

====================
BEHAVIOR RULES
====================

1. Never mention the diagnosis or disease names (caries, pulpitis, gingivitis, stroke, etc.)
   and do NOT give medical explanations, causes, treatments, or advice.

2. Speak like a normal person, using informal, simple language
   ("it hurts", "I noticed", "it feels sharp", "it's annoying", "I'm not sure").
   However:
   - You MUST NOT describe any area, action, or situation as painful, sensitive,
     uncomfortable, weird, or bothering you UNLESS it corresponds to a real symptom
     in your internal list.
   - If the student asks about something that is NOT one of your real symptoms
     (for example cold sensitivity when you have no cold/heat-related symptoms),
     you MUST clearly say that it does NOT bother you or that you have NOT noticed
     any problem with that.

3. In your FIRST message of the conversation, you MUST:
   - Start with a short, natural greeting (e.g. “Hi”, “Hello”, “Hi doctor”).
   - Then briefly and vaguely explain what brought you here today.
   - Do NOT mention many specific symptoms yet.
   - Everything you say must still be consistent with your true symptom list.


4. When the student asks about a SPECIFIC SYMPTOM, you MUST:
   - Identify which real symptom the question corresponds to.
   - If the symptom *is in your true internal list*:
         → answer naturally and set slot_updates[symptom_label] = true.
   - If the symptom is *NOT in your true internal list*:
         → answer clearly that this does NOT cause you problems and set
           slot_updates[symptom_label] = false.
   - Your natural-language answer MUST be consistent with slot_updates:
         if you set a symptom to false, your text must NOT describe that symptom
         as painful, sensitive, uncomfortable, or bothering you.
   - You MUST NOT leave slot_updates empty when the question clearly refers to a symptom.
   - DO NOT say “I'm not sure what you mean” for obvious symptom questions involving:
       pain, hurting, sensitivity, cold, hot, chewing, biting, pressure, swelling,
       taste changes, smell, bleeding, clicking, popping, numbness, electric shock pain,
       headache, facial pressure, jaw opening, jaw closing, etc.
     Only ask for clarification when the question is TRULY ambiguous.

5. If the question is very general ("Do you have any other problems?"),
   you may reveal at most {max_new_clues} NEW real symptom(s).

6. Remain consistent: never contradict your previous answers.

7. Do NOT repeat symptoms you've already mentioned,
   and NEVER invent symptoms or discomforts that are NOT in your real symptom list.

8. Each answer must be short and natural (1–3 sentences).

9. slot_updates RULE:
   - Use true if the asked-about symptom is genuinely present in your internal list.
   - Use false if the student clearly asks about a symptom you do NOT have.
   - If the question is vague, you MAY leave slot_updates empty.
   - The KEYS in slot_updates MUST be the exact English symptom labels
     from your internal list when possible (e.g. "Cold sensitivity", "Pain on chewing",
     "Food impaction pain", "Bad taste / halitosis").
   - If the student describes a symptom using different words (e.g. “cold drink pain”),
     map it to the closest real symptom label.

10. Avoid using the exact phrase “I'm not sure what you mean” repeatedly.
    If clarification is needed, vary your wording politely.

11. If the student talks about unrelated topics (life, jokes, exams, etc.),
    politely redirect back to your symptom discussion.

====================
OUTPUT FORMAT
====================

You MUST return ONLY a valid JSON object, EXACTLY in this structure:

{{
  "assistant_text": "the patient's natural-language answer",
  "slot_updates": {{
    "symptom_name_1": true/false,
    "symptom_name_2": true/false
  }}
}}

Do NOT add any text outside the JSON. No explanations, no comments, no extra keys.
""".strip()




def build_patient_context(state: State, case: Case) -> str:
    """
    Build a context block summarizing what is known so far:
    - short summary
    - confirmed / denied symptoms
    - true symptoms that are not yet confirmed
    (for the model to keep internal consistency).
    """
    confirmed = [k for k, v in state.slots.items() if v is True]
    denied   = [k for k, v in state.slots.items() if v is False]

    all_true_symptoms = sorted(case.symptoms_truth)
    unrevealed_true = [s for s in all_true_symptoms if s not in confirmed]

    context = f"""
CURRENT CONVERSATION CONTEXT:
Short summary: {state.summary or 'The conversation has just started.'}

Symptoms CONFIRMED so far: {', '.join(confirmed) or 'none'}
Symptoms DENIED so far: {', '.join(denied) or 'none'}
True symptoms not yet explicitly mentioned (do NOT reveal them directly to the student):
{', '.join(unrevealed_true) or 'none'}

Instructions:
- Keep your answers coherent with this history.
- Do NOT contradict symptoms you already confirmed or denied.
- Try to be cooperative: if the question clearly refers to one of your symptoms
  (even if phrased informally), answer it instead of saying you don't understand.
- Only ask for clarification when the question is genuinely ambiguous.
""".strip()

    return context



def build_user_prompt(user_msg: str) -> str:
    """
    Wrap the student's message into a clear instruction for the model.
    """
    return f"""
The student asks you: {user_msg}

Follow all the rules from the system and context.

You MUST return ONLY a JSON object with EXACTLY this structure:
{{
  "assistant_text": "your short, realistic answer as the patient",
  "slot_updates": {{
    "some_symptom_key": true/false
  }}
}}
- Use 'slot_updates' to mark specific symptoms that the student is really asking about.
- When they ask about pain or sensitivity with HOT or COLD food or drinks,
  you should normally treat that as a clear question about thermal sensitivity/pain.
- When they ask about chewing, biting, jaw movement, or clicking,
  treat that as a question about function-related symptoms.

Do NOT add any text outside this JSON. No explanations, no comments, no extra keys.
""".strip()


def safe_json_parse(content: str):
    """
    Try to extract and parse a JSON object from the model's response.
    Applies some light cleanup if needed.
    """
    content = content.strip()

    match = re.search(r"\{[\s\S]*\}", content)
    if not match:
        raise RuntimeError(f"Could not find any JSON object in the response:\n{content}")
    json_part = match.group(0)

    json_part = json_part.replace("True", "true").replace("False", "false")
    json_part = json_part.replace("’", "'").replace("„", '"').replace("”", '"')
    json_part = re.sub(r",\s*}", "}", json_part)
    json_part = re.sub(r",\s*]", "]", json_part)

    try:
        return json.loads(json_part)
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON even after cleanup:\n{json_part}\nError: {e}"
        )


def llm_call_groq(
    system_prompt: str,
    context_prompt: str,
    user_prompt: str,
    model: str = "meta-llama/llama-4-scout-17b-16e-instruct",
) -> dict:
    """
    Single call to the Groq LLM.
    We send SYSTEM + CONTEXT + USER blocks all in one 'user' message,
    because the API here uses only 'messages' with roles.
    """
    prompt = (
        f"[SYSTEM]\n{system_prompt}\n\n"
        f"[CONTEXT]\n{context_prompt}\n\n"
        f"[USER]\n{user_prompt}\n"
    )
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=300,
        )
        content = resp.choices[0].message.content.strip()
    except Exception as e:
        raise RuntimeError(f"Groq API error: {e}")

    return safe_json_parse(content)


In [23]:
def step(
    case: Case,
    state: State,
    user_msg: str,
    max_new_clues: int = 1,
    model: str = "meta-llama/llama-4-scout-17b-16e-instruct",
) -> dict:
    """
    Execute one turn of the conversation:
    - the student asks a question (user_msg)
    - the virtual patient (LLM) answers
    - we update the conversation state (summary + slots + history + revealed symptoms).

    Case is compatible with the colleague's structure:
      - case.diagnosis_truth: str
      - case.symptoms_truth: Set[str]
      - case.revealed_symptoms: Set[str]

    Returns a dict with:
      - assistant_text: the patient's answer
      - slot_updates: the updates proposed in this turn
      - summary: updated running summary of the dialogue
      - revealed_symptoms: list of true symptoms that are now considered 'revealed'
    """

    system_prompt = build_patient_system_prompt(case, max_new_clues=max_new_clues)
    context_prompt = build_patient_context(state, case)
    user_prompt = build_user_prompt(user_msg)

    out = llm_call_groq(system_prompt, context_prompt, user_prompt, model=model)

    text = out.get("assistant_text", "").strip()
    slot_updates = out.get("slot_updates", {}) or {}

    true_symptom_norms = {
        s.lower().replace(" ", "_") for s in case.symptoms_truth
    }

    filtered_updates = {}
    for k, v in slot_updates.items():
        k_norm = k.lower().replace(" ", "_")

        if v is False and k_norm in true_symptom_norms:
            continue

        filtered_updates[k] = v

    slot_updates = filtered_updates

    addition = f"Student: {user_msg} | Patient: {text}"
    if state.summary:
        state.summary += " " + addition
    else:
        state.summary = addition

    if isinstance(slot_updates, dict):
        state.slots.update(slot_updates)

    state.history.append({"role": "user", "content": user_msg})
    state.history.append({"role": "assistant", "content": text})

    text_lower = text.lower()
    true_labels = list(case.symptoms_truth)
    label_norm_map = {
        lbl.lower().replace(" ", "_"): lbl
        for lbl in true_labels
    }

    revealed_now = set()

    for lbl in true_labels:
        if lbl.lower() in text_lower:
            revealed_now.add(lbl)

    for k, v in state.slots.items():
        if not v:
            continue
        k_norm = k.lower().replace(" ", "_")
        if k_norm in label_norm_map:
            revealed_now.add(label_norm_map[k_norm])

    case.revealed_symptoms |= revealed_now

    return {
        "assistant_text": text,
        "slot_updates": slot_updates,
        "summary": state.summary,
        "revealed_symptoms": list(case.revealed_symptoms),
    }


In [48]:
"""
Evaluation Metrics for Dental Patient Simulation LLM Agent
Implements SFS, RPCS, and CRS metrics
"""

from dataclasses import dataclass
from typing import List, Set, Dict, Tuple
import re
from collections import defaultdict


@dataclass
class EvaluationResult:
    """Container for evaluation results"""
    sfs: float
    rpcs: float
    crs: float
    details: Dict[str, any]


class SymptomFidelityEvaluator:
    """
    Evaluates how consistently the LLM uses symptoms from knowledge base
    SFS = 1 - (incorrect or missing symptoms / total expected symptoms)
    """

    def __init__(self, mapping: Dict[str, Set[str]]):
        self.mapping = mapping

    def extract_symptoms_from_text(self, text: str, symptom_list: Set[str]) -> Set[str]:
        """Extract mentioned symptoms from LLM output (slightly improved for hyphenated phrases)."""
        text_lower = text.lower()
        mentioned = set()

        for symptom in symptom_list:
            symptom_lower = symptom.lower()

            if symptom_lower in text_lower:
                mentioned.add(symptom)
                continue

            normalized = symptom_lower.replace("-", " ")

            words = [w for w in normalized.split() if w]

            if any(word in text_lower for word in words):
                mentioned.add(symptom)

        return mentioned




    def evaluate(self,
                 diagnosis_key: str,
                 conversation_history: List[Dict[str, str]],
                 revealed_symptoms: Set[str]) -> Tuple[float, Dict]:
        """
        Calculate Symptom Fidelity Score

        Returns:
            (score, details_dict)
        """
        expected_symptoms = self.mapping.get(diagnosis_key, set())

        if not expected_symptoms:
            return 1.0, {"error": "No symptoms defined for diagnosis"}

        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]
        full_text = " ".join(patient_responses)

        mentioned = self.extract_symptoms_from_text(full_text, expected_symptoms)

        hallucinated = self._detect_hallucinations(full_text, expected_symptoms)

        correct = len(mentioned & expected_symptoms)
        missing = len(expected_symptoms - mentioned)
        incorrect = len(hallucinated)

        total_errors = missing + incorrect
        sfs = max(0.0, 1.0 - (total_errors / len(expected_symptoms)))

        details = {
            "expected_count": len(expected_symptoms),
            "correctly_mentioned": correct,
            "missing_symptoms": list(expected_symptoms - mentioned),
            "hallucinated_symptoms": list(hallucinated),
            "revealed_symptoms": list(revealed_symptoms)
        }

        return sfs, details

    def _detect_hallucinations(self, text: str, expected: Set[str]) -> Set[str]:
        """
        Detect symptoms mentioned that aren't in expected set,
        but IGNORE clearly negated mentions like 'no bleeding', 'haven't had any fever', etc.
        """
        symptom_patterns = [
            r"bleeding", r"fever", r"headache", r"nausea",
            r"discharge", r"numbness", r"tingling"
        ]

        hallucinated = set()
        text_lower = text.lower()

        NEGATION_RE = re.compile(
            r"\b(no|not|never|without|haven't|hasn't|didn't|don't|none|no\s+signs\s+of)\b"
        )
        WINDOW = 25

        for pattern in symptom_patterns:
            for match in re.finditer(pattern, text_lower):
                start = match.start()

                left_context_start = max(0, start - WINDOW)
                left_context = text_lower[left_context_start:start]

                if NEGATION_RE.search(left_context):
                    continue

                if not any(re.search(pattern, str(s).lower()) for s in expected):
                    hallucinated.add(pattern)

        return hallucinated



class RolePlayingConsistencyEvaluator:
    """
    Evaluates whether LLM maintains patient persona
    1.0 = Always patient, 0.5 = Occasional breaks, 0.0 = Frequent breaks
    """

    DOCTOR_PATTERNS = [
        r"diagnosis", r"diagnose", r"medical condition",
        r"treatment", r"you should see", r"i recommend",
        r"this could be", r"it might be", r"probably",
        r"consult", r"examination needed"
    ]

    META_PATTERNS = [
        r"as an ai", r"i am a language model", r"i don't have",
        r"i cannot", r"system prompt", r"my role"
    ]

    MEDICAL_ADVICE = [
        r"take.*medication", r"antibiotic", r"prescription",
        r"dental procedure", r"root canal", r"extraction"
    ]

    def evaluate(self, conversation_history: List[Dict[str, str]]) -> Tuple[float, Dict]:
        """
        Calculate Role-Playing Consistency Score

        Returns:
            (score, details_dict)
        """
        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]

        if not patient_responses:
            return 1.0, {"error": "No patient responses found"}

        violations = {
            "doctor_language": 0,
            "meta_awareness": 0,
            "medical_advice": 0
        }

        violation_examples = defaultdict(list)

        for response in patient_responses:
            response_lower = response.lower()

            for pattern in self.DOCTOR_PATTERNS:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["doctor_language"] += len(matches)
                    violation_examples["doctor_language"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

            for pattern in self.META_PATTERNS:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["meta_awareness"] += len(matches)
                    violation_examples["meta_awareness"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

            for pattern in self.MEDICAL_ADVICE:
                matches = re.findall(pattern, response_lower)
                if matches:
                    violations["medical_advice"] += len(matches)
                    violation_examples["medical_advice"].append(
                        f"'{response[:50]}...' contains '{matches[0]}'"
                    )

        total_violations = sum(violations.values())
        total_responses = len(patient_responses)

        if total_violations == 0:
            rpcs = 1.0
        elif total_violations <= total_responses * 0.1:
            rpcs = 0.8
        elif total_violations <= total_responses * 0.3:
            rpcs = 0.5
        else:
            rpcs = 0.2

        details = {
            "total_responses": total_responses,
            "violations": violations,
            "examples": dict(violation_examples),
            "violation_rate": total_violations / total_responses
        }

        return rpcs, details


class ClinicalRealismEvaluator:
    """
    Evaluates realism of patient responses
    Scores 0-4 on multiple dimensions, normalized to 0-1
    """

    PATIENT_LANGUAGE = [
        r"hurts?", r"pain", r"ache", r"sore", r"uncomfortable",
        r"feels?", r"it's", r"when i", r"sometimes", r"usually"
    ]

    MEDICAL_JARGON = [
        r"periodontitis", r"pulpitis", r"necrosis", r"hyperplasia",
        r"pathology", r"etiology", r"prognosis"
    ]

    def evaluate(self, conversation_history: List[Dict[str, str]]) -> Tuple[float, Dict]:
        """
        Calculate Clinical Realism Score

        Dimensions:
        1. Symptom description realism
        2. Appropriate vocabulary
        3. Pain descriptions
        4. Consistency with disease

        Returns:
            (score, details_dict)
        """
        patient_responses = [
            turn['content'] for turn in conversation_history
            if turn['role'] == 'assistant'
        ]

        if not patient_responses:
            return 1.0, {"error": "No patient responses found"}

        full_text = " ".join(patient_responses)
        text_lower = full_text.lower()

        symptom_score = self._score_symptom_descriptions(patient_responses)

        vocab_score = self._score_vocabulary(text_lower)

        pain_score = self._score_pain_descriptions(text_lower)

        consistency_score = self._score_consistency(patient_responses)

        total_score = (symptom_score + vocab_score + pain_score + consistency_score) / 16.0

        details = {
            "symptom_description": symptom_score / 4.0,
            "vocabulary": vocab_score / 4.0,
            "pain_description": pain_score / 4.0,
            "consistency": consistency_score / 4.0,
            "dimensions_raw": {
                "symptom": symptom_score,
                "vocab": vocab_score,
                "pain": pain_score,
                "consistency": consistency_score
            }
        }

        return total_score, details

    def _score_symptom_descriptions(self, responses: List[str]) -> int:
        """Score how realistically symptoms are described (0-4)."""
        text = " ".join(r.lower() for r in responses)

        subjective_patterns = [
            "i feel",
            "it feels",
            "feels like",
            "i've been having",
            "i've been dealing with",
            "it's been bothering me",
            "it bothers me",
            "it's been hurting",
            "it's been annoying",
        ]

        concrete_patterns = [
            "when i",
            "when i'm",
            "when im",
            "after i",
            "if i",

            "when i eat",
            "when i'm eating",
            "when i drink",
            "when i'm drinking",
            "when i brush",
            "when i bite",
            "when i chew",
            "when i tap on",
            "when i tap the tooth",
        ]

        subjective_count = sum(1 for p in subjective_patterns if p in text)
        concrete_count = sum(1 for p in concrete_patterns if p in text)

        score = 0
        if subjective_count > 0:
            score += 2
        if concrete_count > 0:
            score += 2

        return min(4, score)


    def _score_vocabulary(self, text: str) -> int:
        """Score appropriateness of vocabulary (0-4)"""
        patient_lang = sum(
            len(re.findall(pattern, text))
            for pattern in self.PATIENT_LANGUAGE
        )

        jargon = sum(
            len(re.findall(pattern, text))
            for pattern in self.MEDICAL_JARGON
        )

        if jargon > 2:
            return 0
        elif jargon > 0:
            return 2
        elif patient_lang >= 5:
            return 4
        elif patient_lang >= 2:
            return 3
        else:
            return 1

    def _score_pain_descriptions(self, text: str) -> int:
        """Score quality of pain descriptions (0-4)."""
        score = 0
        text = text.lower()

        if re.search(r"(back|front|side|top|bottom|left|right|upper|lower)", text):
            score += 1

        if re.search(r"(sharp|dull|throbbing|constant|mild|severe|strong|intense)", text):
            score += 1

        if re.search(r"(when|after|during|while)", text):
            score += 1

        if re.search(
            r"(minute|hour|day|week|always|sometimes|"
            r"for a while|for some time|for a few days|for a few weeks|"
            r"for weeks|for months|on and off|all the time|"
            r"it's been bothering me for|i've been having.*for)",
            text,
        ):
            score += 1

        return score


    def _score_consistency(self, responses: List[str]) -> int:
        """Score consistency across responses (0-4)"""
        if len(responses) < 2:
            return 4

        score = 4

        pain_keywords = set()
        for response in responses:
            response_lower = response.lower()
            if "pain" in response_lower:
                for word in ["sharp", "dull", "throbbing", "aching"]:
                    if word in response_lower:
                        pain_keywords.add(word)

        if len(pain_keywords) > 2:
            score -= 1

        return max(0, score)


class PatientSimulationEvaluator:
    """Main evaluator combining all metrics"""

    def __init__(self, mapping: Dict[str, Set[str]]):
        self.sfs_evaluator = SymptomFidelityEvaluator(mapping)
        self.rpcs_evaluator = RolePlayingConsistencyEvaluator()
        self.crs_evaluator = ClinicalRealismEvaluator()

    def evaluate(self,
                 case,
                 state) -> EvaluationResult:
        """
        Evaluate a complete conversation

        Args:
            case: Case object with diagnosis_truth and symptoms_truth
            state: State object with conversation history

        Returns:
            EvaluationResult with all metrics
        """
        sfs, sfs_details = self.sfs_evaluator.evaluate(
            case.diagnosis_truth,
            state.history,
            case.revealed_symptoms
        )

        rpcs, rpcs_details = self.rpcs_evaluator.evaluate(state.history)

        crs, crs_details = self.crs_evaluator.evaluate(state.history)

        all_details = {
            "sfs": sfs_details,
            "rpcs": rpcs_details,
            "crs": crs_details,
            "conversation_length": len(state.history),
            "diagnosis": case.diagnosis_truth
        }

        return EvaluationResult(
            sfs=sfs,
            rpcs=rpcs,
            crs=crs,
            details=all_details
        )

    def print_report(self, result: EvaluationResult):
        """Print a formatted evaluation report"""
        print("\n" + "="*60)
        print("PATIENT SIMULATION EVALUATION REPORT")
        print("="*60)

        print(f"\n📊 OVERALL SCORES:")
        print(f"   Symptom Fidelity Score (SFS):      {result.sfs:.3f}")
        print(f"   Role-Playing Consistency (RPCS):   {result.rpcs:.3f}")
        print(f"   Clinical Realism Score (CRS):      {result.crs:.3f}")
        print(f"   Average Score:                      {(result.sfs + result.rpcs + result.crs) / 3:.3f}")

        print(f"\n🔍 DETAILED BREAKDOWN:")

        # SFS details
        print(f"\n   Symptom Fidelity:")
        sfs_d = result.details['sfs']
        print(f"   - Expected symptoms: {sfs_d['expected_count']}")
        print(f"   - Correctly mentioned: {sfs_d['correctly_mentioned']}")
        if sfs_d['missing_symptoms']:
            print(f"   - Missing: {', '.join(sfs_d['missing_symptoms'][:3])}")
        if sfs_d['hallucinated_symptoms']:
            print(f"   - Hallucinated: {', '.join(sfs_d['hallucinated_symptoms'])}")

        # RPCS details
        print(f"\n   Role-Playing Consistency:")
        rpcs_d = result.details['rpcs']
        print(f"   - Total responses: {rpcs_d['total_responses']}")
        print(f"   - Violation rate: {rpcs_d['violation_rate']:.1%}")
        for vtype, count in rpcs_d['violations'].items():
            if count > 0:
                print(f"   - {vtype}: {count} violations")

        # CRS details
        print(f"\n   Clinical Realism:")
        crs_d = result.details['crs']
        print(f"   - Symptom description: {crs_d['symptom_description']:.2f}")
        print(f"   - Vocabulary: {crs_d['vocabulary']:.2f}")
        print(f"   - Pain description: {crs_d['pain_description']:.2f}")
        print(f"   - Consistency: {crs_d['consistency']:.2f}")

        print("\n" + "="*60 + "\n")


def evaluate_conversation(case, state, mapping):
    """
    Convenience function to evaluate a conversation

    Args:
        case: Case object from your simulation
        state: State object with conversation history
        mapping: The disease->symptoms mapping dictionary

    Returns:
        EvaluationResult
    """
    evaluator = PatientSimulationEvaluator(mapping)
    result = evaluator.evaluate(case, state)
    evaluator.print_report(result)
    return result

In [28]:
"""
Comprehensive Test Suite for Dental Patient Simulation LLM
Run this in your Colab notebook after loading the model
"""

import random
from typing import List, Dict, Tuple
import json
from datetime import datetime

# ============================================================================
# REALISTIC DENTAL CONSULTATION QUESTIONS
# ============================================================================
CONSULTATION_QUESTIONS = {
    "opening": [
        "Good morning! What brings you to the dental clinic today?",
        "Hello, I'm Dr. Smith. Can you tell me what's been bothering you?",
        "Hi there, what seems to be the problem with your teeth?",
        "Welcome to the clinic. What can I help you with today?",
    ],

    "chief_complaint": [
        "Can you describe the pain you're experiencing in more detail?",
        "Where exactly in your mouth is the pain located? Can you point to the specific tooth or area?",
        "How long have you been experiencing these symptoms?",
        "When did you first notice this problem?",
        "Is this the first time you've had this issue, or has it happened before?",
    ],

    "pain_characteristics": [
        "On a scale of 1 to 10, with 10 being the worst pain imaginable, how would you rate your pain right now?",
        "How would you describe the pain? Is it sharp, dull, throbbing, constant, or does it come and go?",
        "Does the pain spread to other areas, like your jaw, ear, or head?",
        "Is the pain worse at any particular time of day - morning, afternoon, or night?",
        "Does the pain wake you up at night, or does it affect your sleep?",
    ],

    "triggers_and_relievers": [
        "What makes the pain worse? For example, eating, drinking, or touching the area?",
        "Does hot food or drinks cause any problems? What about cold things like ice cream or cold water?",
        "Do sweet foods or drinks trigger the pain or make it worse?",
        "What about chewing - does biting down on food cause pain?",
        "Have you found anything that helps relieve the pain or make it more bearable?",
        "Does the pain get better when you take painkillers like ibuprofen or paracetamol?",
    ],

    "associated_symptoms": [
        "Have you noticed any swelling in your gums or face?",
        "Is there any bleeding when you brush your teeth or eat?",
        "Do you have any unusual taste in your mouth, perhaps a bad or salty taste?",
        "Have you noticed any bad breath or odor coming from your mouth?",
        "Is there any discharge or pus coming from your gums?",
        "Have you had any fever or felt generally unwell?",
        "Do you have any difficulty opening your mouth fully?",
        "Does it hurt when you swallow or move your jaw?",
    ],

    "specific_observations": [
        "Have you noticed if the tooth feels loose or moves when you touch it?",
        "Does the tooth feel higher than your other teeth when you bite down?",
        "Can you see any visible holes, dark spots, or broken pieces on your teeth?",
        "Have you noticed any changes in the color of your teeth or gums?",
        "Is there a particular tooth that hurts when you tap on it with your finger?",
    ],

    "impact_on_daily_life": [
        "Is the pain affecting your ability to eat or drink normally?",
        "Are you avoiding chewing on one side of your mouth because of the pain?",
        "Has this problem affected your work or daily activities?",
        "Are you able to brush your teeth normally, or does that area hurt too much?",
    ],

    "follow_up": [
        "Is there anything else that you've noticed or that's concerning you?",
        "Are there any other symptoms you'd like to tell me about?",
        "Have you tried anything at home to help with this problem?",
        "Is there anything you'd like to ask me about your symptoms?",
    ]
}

# ============================================================================
# DISEASE-SPECIFIC QUESTION SETS
# ============================================================================
DISEASE_SPECIFIC_QUESTIONS = {
    "periodontal_abscess": [
        "Is there a specific area on your gum that looks swollen or feels like a bump?",
        "When you press on the swollen area, does it hurt more?",
        "Have you noticed any liquid or drainage coming from your gums?",
        "Does it feel like the pain is coming from the gum rather than the tooth itself?",
    ],

    "simple_cavities": [
        "Does the pain only last for a few seconds when you eat or drink something cold or sweet?",
        "After the painful sensation, does it go away quickly or linger?",
        "Can you see any dark spots or holes in your teeth when you look in the mirror?",
    ],

    "acute_apical_periodontitis": [
        "Does it hurt specifically when you bite down or chew food?",
        "When you tap the tooth with your finger, is it very sensitive?",
        "Does the tooth feel like it's sitting higher or sticking out more than your other teeth?",
        "Is the tooth loose or does it move slightly when you touch it?",
    ],

    "pericoronitis": [
        "Is the pain near your wisdom tooth at the back of your mouth?",
        "Can you open your mouth all the way, or is it difficult?",
        "Is it painful when you swallow food or saliva?",
        "Can you see the gum covering part of your wisdom tooth, and does it look red or swollen?",
    ],

    "reversible_pulpitis": [
        "Does the pain only happen when something cold touches the tooth?",
        "Once you remove the cold drink or food, does the pain stop immediately?",
        "Do you ever have pain when you're not eating or drinking - like spontaneous pain?",
    ],

    "irreversible_pulpitis": [
        "Do you get sudden, severe pain that happens even when you're not eating or drinking?",
        "Does the pain last for several minutes or longer after it starts?",
        "Does heat make it worse - like drinking hot coffee or tea?",
        "Does the pain sometimes shoot up to your ear or spread to your head?",
        "Does cold actually help relieve the pain, or does it make it worse?",
    ],

    "pulp_necrosis": [
        "Did you have severe pain before, but now the pain has changed or become different?",
        "Does heat cause severe, immediate pain that lasts a while?",
        "When you bite down, does it feel like there's pressure building up in the tooth?",
        "If I were to tap on the tooth, would that be very painful?",
    ]
}

# ============================================================================
# TEST CONVERSATION BUILDER
# ============================================================================
class TestConversationBuilder:
    """Builds realistic test conversations"""

    def __init__(self):
        self.question_pool = CONSULTATION_QUESTIONS
        self.disease_questions = DISEASE_SPECIFIC_QUESTIONS

    def build_short_conversation(self, disease_key: str = None) -> List[str]:
        """Build a short 5-7 question conversation"""
        questions = []

        questions.append(random.choice(self.question_pool["opening"]))

        questions.append(random.choice(self.question_pool["chief_complaint"]))

        questions.append(random.choice(self.question_pool["pain_characteristics"]))

        questions.append(random.choice(self.question_pool["triggers_and_relievers"]))

        questions.append(random.choice(self.question_pool["associated_symptoms"]))

        if disease_key and disease_key in self.disease_questions:
            questions.append(random.choice(self.disease_questions[disease_key]))

        questions.append(random.choice(self.question_pool["follow_up"]))

        return questions

    def build_medium_conversation(self, disease_key: str = None) -> List[str]:
        """Build a medium 10-12 question conversation"""
        questions = []

        questions.append(random.choice(self.question_pool["opening"]))

        questions.extend(random.sample(self.question_pool["chief_complaint"], 2))

        questions.extend(random.sample(self.question_pool["pain_characteristics"], 2))

        questions.extend(random.sample(self.question_pool["triggers_and_relievers"], 2))

        questions.extend(random.sample(self.question_pool["associated_symptoms"], 2))

        questions.append(random.choice(self.question_pool["specific_observations"]))

        if disease_key and disease_key in self.disease_questions:
            questions.extend(random.sample(self.disease_questions[disease_key], min(2, len(self.disease_questions[disease_key]))))

        questions.append(random.choice(self.question_pool["follow_up"]))

        return questions

    def build_long_conversation(self, disease_key: str = None) -> List[str]:
        """Build a long 15-20 question conversation"""
        questions = []

        questions.append(random.choice(self.question_pool["opening"]))

        for phase in ["chief_complaint", "pain_characteristics", "triggers_and_relievers",
                      "associated_symptoms", "specific_observations", "impact_on_daily_life"]:
            num_questions = 3 if phase in ["pain_characteristics", "triggers_and_relievers", "associated_symptoms"] else 2
            available = self.question_pool[phase]
            questions.extend(random.sample(available, min(num_questions, len(available))))

        if disease_key and disease_key in self.disease_questions:
            questions.extend(self.disease_questions[disease_key])

        questions.extend(random.sample(self.question_pool["follow_up"], 2))

        return questions

# ============================================================================
# AUTOMATED TEST RUNNER
# ============================================================================

class PatientSimulationTester:
    """Automated tester for patient simulation LLM"""

    def __init__(self, model, tokenizer, mapping, step_function):
        """
        Args:
            model: Your loaded LLM model
            tokenizer: Tokenizer for the model
            mapping: Disease -> symptoms mapping
            step_function: Your step() function from notebook
        """
        self.model = model
        self.tokenizer = tokenizer
        self.mapping = mapping
        self.step = step_function
        self.conversation_builder = TestConversationBuilder()

    def run_single_test(self,
                    disease_key: str = None,
                    conversation_length: str = "medium",
                    verbose: bool = True) -> Tuple[Case, State]:
      """
      Run a single test conversation

      Args:
          disease_key: Specific disease to test (None for random)
          conversation_length: "short", "medium", or "long"
          verbose: Print conversation as it happens

      Returns:
          (case, state) tuple
      """

      if disease_key is None:
          disease_key = random.choice(list(self.mapping.keys()))

      symptoms = set(self.mapping[disease_key])
      case = Case(
          diagnosis_truth=disease_key,
          symptoms_truth=symptoms,
      )
      state = State()

      if conversation_length == "short":
          questions = self.conversation_builder.build_short_conversation(disease_key)
      elif conversation_length == "long":
          questions = self.conversation_builder.build_long_conversation(disease_key)
      else:
          questions = self.conversation_builder.build_medium_conversation(disease_key)

      if verbose:
          print(f"\n{'='*70}")
          print(f"Testing: {disease_key.replace('_', ' ').title()}")
          print(f"Expected symptoms: {len(symptoms)}")
          print(f"Conversation length: {conversation_length} ({len(questions)} questions)")
          print(f"{'='*70}\n")

      for i, question in enumerate(questions, 1):
          if verbose:
              print(f"[Q{i}] Doctor: {question}")

          try:
              out = self.step(case, state, question)
              if isinstance(out, dict):
                  response_text = out.get("assistant_text", "")
              else:
                  response_text = str(out)

              if verbose:
                  print(f"[A{i}] Patient: {response_text}\n")

          except Exception as e:
              print(f"❌ Error at question {i}: {e}")
              break

      return case, state


    def run_test_suite(self,
                      num_tests_per_disease: int = 2,
                      conversation_length: str = "medium",
                      save_results: bool = True) -> Dict:
        """
        Run comprehensive test suite across all diseases
        """

        evaluator = PatientSimulationEvaluator(self.mapping)

        all_results = {
            "test_info": {
                "timestamp": datetime.now().isoformat(),
                "num_tests_per_disease": num_tests_per_disease,
                "conversation_length": conversation_length,
                "total_diseases": len(self.mapping)
            },
            "disease_results": {},
            "aggregate_metrics": {}
        }

        print(f"\n{'='*70}")
        print(f"STARTING TEST SUITE")
        print(f"{'='*70}")
        print(f"Testing {len(self.mapping)} diseases")
        print(f"{num_tests_per_disease} conversations per disease")
        print(f"Conversation length: {conversation_length}")
        print(f"Total tests: {len(self.mapping) * num_tests_per_disease}\n")

        all_scores = {"sfs": [], "rpcs": [], "crs": []}

        for disease_idx, disease_key in enumerate(self.mapping.keys(), 1):
            print(f"\n[{disease_idx}/{len(self.mapping)}] Testing: {disease_key.replace('_', ' ').title()}")
            print("-" * 70)

            disease_results = []

            for test_num in range(num_tests_per_disease):
                print(f"\n  Test {test_num + 1}/{num_tests_per_disease}...")

                case, state = self.run_single_test(
                    disease_key=disease_key,
                    conversation_length=conversation_length,
                    verbose=False
                )

                result = evaluator.evaluate(case, state)

                disease_results.append({
                    "sfs": result.sfs,
                    "rpcs": result.rpcs,
                    "crs": result.crs,
                    "details": result.details
                })

                all_scores["sfs"].append(result.sfs)
                all_scores["rpcs"].append(result.rpcs)
                all_scores["crs"].append(result.crs)

                print(f"    SFS: {result.sfs:.3f} | RPCS: {result.rpcs:.3f} | CRS: {result.crs:.3f}")

            avg_sfs = sum(r["sfs"] for r in disease_results) / len(disease_results)
            avg_rpcs = sum(r["rpcs"] for r in disease_results) / len(disease_results)
            avg_crs = sum(r["crs"] for r in disease_results) / len(disease_results)

            all_results["disease_results"][disease_key] = {
                "average_scores": {
                    "sfs": avg_sfs,
                    "rpcs": avg_rpcs,
                    "crs": avg_crs,
                    "overall": (avg_sfs + avg_rpcs + avg_crs) / 3
                },
                "individual_tests": disease_results
            }

            print(f"\n  Disease Average: SFS={avg_sfs:.3f}, RPCS={avg_rpcs:.3f}, CRS={avg_crs:.3f}")

        all_results["aggregate_metrics"] = {
            "sfs": {
                "mean": sum(all_scores["sfs"]) / len(all_scores["sfs"]),
                "min": min(all_scores["sfs"]),
                "max": max(all_scores["sfs"]),
                "std": self._std(all_scores["sfs"])
            },
            "rpcs": {
                "mean": sum(all_scores["rpcs"]) / len(all_scores["rpcs"]),
                "min": min(all_scores["rpcs"]),
                "max": max(all_scores["rpcs"]),
                "std": self._std(all_scores["rpcs"])
            },
            "crs": {
                "mean": sum(all_scores["crs"]) / len(all_scores["crs"]),
                "min": min(all_scores["crs"]),
                "max": max(all_scores["crs"]),
                "std": self._std(all_scores["crs"])
            }
        }

        self._print_final_report(all_results)

        if save_results:
            filename = f"test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(all_results, f, indent=2, default=str)
            print(f"\n✅ Results saved to: {filename}")

        return all_results

    def _std(self, values):
        """Calculate standard deviation"""
        import math
        if len(values) == 0:
            return 0
        mean = sum(values) / len(values)
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        return math.sqrt(variance)

    def _print_final_report(self, results):
        """Print comprehensive final report"""
        print(f"\n\n{'='*70}")
        print("FINAL TEST SUITE REPORT")
        print(f"{'='*70}\n")

        agg = results["aggregate_metrics"]

        print("📊 OVERALL PERFORMANCE:")
        print(f"   Symptom Fidelity Score (SFS)")
        print(f"      Mean: {agg['sfs']['mean']:.3f} ± {agg['sfs']['std']:.3f}")
        print(f"      Range: [{agg['sfs']['min']:.3f}, {agg['sfs']['max']:.3f}]")
        print()
        print(f"   Role-Playing Consistency (RPCS)")
        print(f"      Mean: {agg['rpcs']['mean']:.3f} ± {agg['rpcs']['std']:.3f}")
        print(f"      Range: [{agg['rpcs']['min']:.3f}, {agg['rpcs']['max']:.3f}]")
        print()
        print(f"   Clinical Realism Score (CRS)")
        print(f"      Mean: {agg['crs']['mean']:.3f} ± {agg['crs']['std']:.3f}")
        print(f"      Range: [{agg['crs']['min']:.3f}, {agg['crs']['max']:.3f}]")

        overall_avg = (agg['sfs']['mean'] + agg['rpcs']['mean'] + agg['crs']['mean']) / 3
        print(f"\n   🎯 OVERALL AVERAGE: {overall_avg:.3f}")

        print(f"\n\n📈 PERFORMANCE BY DISEASE:\n")

        disease_scores = []
        for disease, data in results["disease_results"].items():
            score = data["average_scores"]["overall"]
            disease_scores.append((disease, score, data["average_scores"]))

        disease_scores.sort(key=lambda x: x[1], reverse=True)

        print("   Top 3 Best Performing:")
        for i, (disease, score, scores) in enumerate(disease_scores[:3], 1):
            print(f"      {i}. {disease.replace('_', ' ').title()}: {score:.3f}")
            print(f"         (SFS: {scores['sfs']:.3f}, RPCS: {scores['rpcs']:.3f}, CRS: {scores['crs']:.3f})")

        print("\n   Bottom 3 Need Improvement:")
        for i, (disease, score, scores) in enumerate(disease_scores[-3:], 1):
            print(f"      {i}. {disease.replace('_', ' ').title()}: {score:.3f}")
            print(f"         (SFS: {scores['sfs']:.3f}, RPCS: {scores['rpcs']:.3f}, CRS: {scores['crs']:.3f})")

        print(f"\n{'='*70}\n")


# ============================================================================
# QUICK START FUNCTIONS
# ============================================================================

def quick_test(model, tokenizer, mapping, step_function,
               num_conversations: int = 5):
    """
    Quick test - runs a few conversations and shows results

    Usage:
        quick_test(model, tokenizer, mapping, step, num_conversations=5)
    """
    tester = PatientSimulationTester(model, tokenizer, mapping, step_function)

    print("🚀 Running Quick Test...")
    print(f"   {num_conversations} random conversations\n")

    evaluator = PatientSimulationEvaluator(mapping)

    results = []
    for i in range(num_conversations):
        case, state = tester.run_single_test(verbose=True)
        result = evaluator.evaluate(case, state)
        evaluator.print_report(result)
        results.append(result)

    avg_sfs = sum(r.sfs for r in results) / len(results)
    avg_rpcs = sum(r.rpcs for r in results) / len(results)
    avg_crs = sum(r.crs for r in results) / len(results)

    print(f"\n{'='*70}")
    print(f"QUICK TEST SUMMARY")
    print(f"{'='*70}")
    print(f"Average SFS:  {avg_sfs:.3f}")
    print(f"Average RPCS: {avg_rpcs:.3f}")
    print(f"Average CRS:  {avg_crs:.3f}")
    print(f"Overall:      {(avg_sfs + avg_rpcs + avg_crs) / 3:.3f}")
    print(f"{'='*70}\n")


def full_test_suite(model, tokenizer, mapping, step_function,
                   tests_per_disease: int = 3,
                   length: str = "medium"):
    """
    Full test suite - comprehensive evaluation of all diseases

    Usage:
        results = full_test_suite(model, tokenizer, mapping, step,
                                 tests_per_disease=3, length="medium")
    """
    tester = PatientSimulationTester(model, tokenizer, mapping, step_function)
    return tester.run_test_suite(
        num_tests_per_disease=tests_per_disease,
        conversation_length=length,
        save_results=True
    )

In [58]:
"""
RUN TESTS FOR YOUR VIRTUAL PATIENT (GROQ-BASED)
Assumes you already defined:
- mapping: Dict[str, Set[str]]  (disease -> set of symptom strings)
- Case, State dataclasses
- step(case, state, user_msg, ...) -> dict with "assistant_text"
- PatientSimulationEvaluator
- PatientSimulationTester, quick_test, full_test_suite
"""

# ============================================================================
# Run a Quick Test (5 conversations)
# ============================================================================

print("="*70)
print("OPTION 1: QUICK TEST")
print("="*70)
print("Runs 5 random conversations with your LLM and evaluates them")
print()

# quick_test(None, None, mapping, step, num_conversations=5)


# ============================================================================
# Run Full Test Suite (all diseases, multiple tests each)
# ============================================================================

print("\n" + "="*70)
print("OPTION 2: FULL TEST SUITE")
print("="*70)
print("Tests all diseases in your mapping with multiple conversations each")
print("Generates comprehensive report + saves to JSON")
print()

results = full_test_suite(
    None,
    None,
    mapping,
    step,
    tests_per_disease=2,
    length="medium"
)


# ============================================================================
# Test a Specific Disease
# ============================================================================

print("\n" + "="*70)
print("OPTION 3: TEST SPECIFIC DISEASE")
print("="*70)
print("Test one specific disease in detail")
print()

# from typing import Set
#
# tester = PatientSimulationTester(None, None, mapping, step)
# evaluator = PatientSimulationEvaluator(mapping)
#
# case, state = tester.run_single_test(
#     disease_key="periodontal_abscess",  # cheie din mapping
#     conversation_length="long",         # "short", "medium", "long"
#     verbose=True
# )
#
# # Evaluezi conversația
# result = evaluator.evaluate(case, state)
# evaluator.print_report(result)


# ============================================================================
# Custom Test with Your Own Questions
# ============================================================================

print("\n" + "="*70)
print("OPTION 4: CUSTOM QUESTIONS")
print("="*70)
print("Test with your own specific questions")
print()

#
# disease_key = "irreversible_pulpitis"
# symptoms = set(mapping[disease_key])
# case = Case(diagnosis_truth=disease_key, symptoms_truth=symptoms)
# state = State()
#
# my_questions = [
#     "What brings you here today?",
#     "Where exactly does it hurt?",
#     "Does cold water make it worse or better?",
#     "Do you get pain even when you're not eating?",
#     "Does the pain radiate to your ear?",
#     "How long does the pain typically last?",
#     "What have you tried to relieve the pain?"
# ]
#
# for question in my_questions:
#     out = step(case, state, question)
#     answer = out.get("assistant_text", "")
#     print(f"Q: {question}")
#     print(f"A: {answer}\n")
#
# evaluator = PatientSimulationEvaluator(mapping)
# result = evaluator.evaluate(case, state)
# evaluator.print_report(result)


# ============================================================================
# RECOMMENDED WORKFLOW
# ============================================================================

print("\n" + "="*70)
print("RECOMMENDED TESTING WORKFLOW")
print("="*70)
print("""
1. Start with QUICK TEST (5 conversations)
   - Get a feel for how your model performs
   - See if there are obvious issues

2. If Quick Test looks good, run FULL TEST SUITE
   - Tests all diseases systematically
   - Gives you comprehensive metrics
   - Saves detailed JSON for analysis

3. Look at the results:
   - Which diseases perform well?
   - Which need improvement?
   - Is SFS low? → Model isn't using correct symptoms
   - Is RPCS low? → Model breaks character
   - Is CRS low? → Responses aren't realistic

4. Deep dive into problem areas:
   - Use OPTION 3 to test specific diseases
   - Check the detailed violation examples
   - Review missing/hallucinated symptoms

5. Iterate:
   - Adjust your system prompts
   - (Optional) adjust mapping or evaluation rules
   - Re-run tests to measure improvement
""")

print("="*70)
print("Ready to test! Uncomment one of the options above to start.")
print("="*70)


OPTION 1: QUICK TEST
Runs 5 random conversations with your LLM and evaluates them


OPTION 2: FULL TEST SUITE
Tests all diseases in your mapping with multiple conversations each
Generates comprehensive report + saves to JSON


STARTING TEST SUITE
Testing 8 diseases
2 conversations per disease
Conversation length: medium
Total tests: 16


[1/8] Testing: Periodontal Abscess
----------------------------------------------------------------------

  Test 1/2...
    SFS: 0.875 | RPCS: 0.800 | CRS: 0.938

  Test 2/2...
    SFS: 0.875 | RPCS: 1.000 | CRS: 0.938

  Disease Average: SFS=0.875, RPCS=0.900, CRS=0.938

[2/8] Testing: Simple Cavities
----------------------------------------------------------------------

  Test 1/2...
    SFS: 1.000 | RPCS: 0.800 | CRS: 1.000

  Test 2/2...
    SFS: 1.000 | RPCS: 0.800 | CRS: 1.000

  Disease Average: SFS=1.000, RPCS=0.800, CRS=1.000

[3/8] Testing: Acute Apical Periodontitis
----------------------------------------------------------------------

  